In [1]:
import duckdb
from pathlib import Path
import networkx as nx
import numpy as np
import faiss
import pickle
import torch

from transformers import AutoTokenizer, AutoModel


In [2]:
class HFTextEmbedder:
    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()

    @torch.no_grad()
    def encode(self, texts, normalize_embeddings=True):
        if isinstance(texts, str):
            texts = [texts]

        inputs = self.tokenizer(
            texts, padding=True, truncation=True, return_tensors="pt"
        )
        outputs = self.model(**inputs)
        emb = outputs.last_hidden_state.mean(dim=1).cpu().numpy()

        if normalize_embeddings:
            emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)

        return emb.astype("float32")


In [3]:
def build_schema_artifacts(sqlite_path: Path):
    con = duckdb.connect(database=":memory:")
    con.execute("INSTALL sqlite;")
    con.execute("LOAD sqlite;")
    con.execute(f"ATTACH DATABASE '{sqlite_path}' AS db (TYPE sqlite);")

    tables = con.execute("""
        SELECT table_name
        FROM duckdb_tables()
        WHERE database_name = 'db';
    """).fetchall()

    table_names = [t[0] for t in tables]

    schema = {}
    for table in table_names:
        cols = con.execute(f"""
            SELECT column_name, data_type
            FROM duckdb_columns()
            WHERE database_name = 'db'
              AND table_name = '{table}';
        """).fetchall()

        schema[table] = [{"column": c[0], "type": c[1]} for c in cols]

    # --- Graph ---
    G = nx.Graph()
    for table in schema:
        G.add_node(table, type="table")

        for col in schema[table]:
            col_node = f"{table}.{col['column']}"
            G.add_node(col_node, type="column")
            G.add_edge(table, col_node, relation="has_column")

    # --- Foreign keys ---
    for table in table_names:
        try:
            fks = con.execute(
                f"PRAGMA db.foreign_key_list('{table}');"
            ).fetchall()

            for fk in fks:
                src = f"{table}.{fk[3]}"
                tgt = f"{fk[2]}.{fk[4]}"
                if src in G and tgt in G:
                    G.add_edge(src, tgt, relation="FK")
        except:
            pass

    # --- Text corpus ---
    texts, ids = [], []
    for table, cols in schema.items():
        texts.append(
            f"table {table} with columns " +
            ", ".join(c["column"] for c in cols)
        )
        ids.append(table)

        for c in cols:
            texts.append(
                f"column {c['column']} in table {table} of type {c['type']}"
            )
            ids.append(f"{table}.{c['column']}")

    con.close()
    return G, texts, ids


In [4]:
SPIDER_DB_ROOT = Path("../data/spider/database")
SCHEMA_OUT = Path("../schemas/spider")
SCHEMA_OUT.mkdir(parents=True, exist_ok=True)

embedder = HFTextEmbedder()

for db_dir in SPIDER_DB_ROOT.iterdir():
    if not db_dir.is_dir():
        continue

    db_id = db_dir.name
    sqlite_file = db_dir / f"{db_id}.sqlite"
    if not sqlite_file.exists():
        continue

    print(f"⏳ Processing DB: {db_id}")

    G, texts, ids = build_schema_artifacts(sqlite_file)

    embeddings = embedder.encode(texts)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    artifact = {
        "graph": G,
        "schema_texts": texts,
        "schema_ids": ids,
        "faiss_index": index
    }

    with open(SCHEMA_OUT / f"{db_id}.pkl", "wb") as f:
        pickle.dump(artifact, f)

print("✅ ALL SPIDER SCHEMA GRAPHS BUILT")


⏳ Processing DB: academic
⏳ Processing DB: activity_1
⏳ Processing DB: aircraft
⏳ Processing DB: allergy_1
⏳ Processing DB: apartment_rentals
⏳ Processing DB: architecture
⏳ Processing DB: assets_maintenance
⏳ Processing DB: baseball_1
⏳ Processing DB: battle_death
⏳ Processing DB: behavior_monitoring
⏳ Processing DB: bike_1
⏳ Processing DB: body_builder
⏳ Processing DB: book_2
⏳ Processing DB: browser_web
⏳ Processing DB: candidate_poll
⏳ Processing DB: car_1
⏳ Processing DB: chinook_1
⏳ Processing DB: cinema
⏳ Processing DB: city_record
⏳ Processing DB: climbing
⏳ Processing DB: club_1
⏳ Processing DB: coffee_shop
⏳ Processing DB: college_1
⏳ Processing DB: college_2
⏳ Processing DB: college_3
⏳ Processing DB: company_1
⏳ Processing DB: company_employee
⏳ Processing DB: company_office
⏳ Processing DB: concert_singer
⏳ Processing DB: county_public_safety
⏳ Processing DB: course_teach
⏳ Processing DB: cre_Docs_and_Epenses
⏳ Processing DB: cre_Doc_Control_Systems
⏳ Processing DB: cre_Do

In [5]:
# -------------------------------
# BUILD ALL BIRD SCHEMA GRAPHS
# -------------------------------

BIRD_DB_ROOT = Path("../data/bird/databases/dev_databases") 
BIRD_SCHEMA_OUT = Path("../schemas/bird")
BIRD_SCHEMA_OUT.mkdir(parents=True, exist_ok=True)

for db_dir in BIRD_DB_ROOT.iterdir():
    if not db_dir.is_dir():
        continue

    db_id = db_dir.name
    sqlite_file = db_dir / f"{db_id}.sqlite"

    if not sqlite_file.exists():
        continue

    print(f"⏳ Processing BIRD DB: {db_id}")

    try:
        G, texts, ids = build_schema_artifacts(sqlite_file)

        embeddings = embedder.encode(texts)
        index = faiss.IndexFlatIP(embeddings.shape[1])
        index.add(embeddings)

        artifact = {
            "graph": G,
            "schema_texts": texts,
            "schema_ids": ids,
            "faiss_index": index
        }

        with open(BIRD_SCHEMA_OUT / f"{db_id}.pkl", "wb") as f:
            pickle.dump(artifact, f)

    except Exception as e:
        print(f"⚠️ Skipping BIRD DB {db_id}: {e}")

print("✅ ALL BIRD SCHEMA GRAPHS BUILT")


⏳ Processing BIRD DB: california_schools
⏳ Processing BIRD DB: card_games
⏳ Processing BIRD DB: codebase_community
⏳ Processing BIRD DB: debit_card_specializing
⏳ Processing BIRD DB: european_football_2
⏳ Processing BIRD DB: financial
⏳ Processing BIRD DB: formula_1
⏳ Processing BIRD DB: student_club
⏳ Processing BIRD DB: superhero
⏳ Processing BIRD DB: thrombosis_prediction
⏳ Processing BIRD DB: toxicology
✅ ALL BIRD SCHEMA GRAPHS BUILT
